# Homework 2: training pipeline

This code will test your homework 2 solutions by using them in a complete ML pipeline. You should run this code in order to tune your model and save your model weights (which will also be uploaded as part of your solution)

In [91]:
# Download the training data from the homework2 folder:
# unzip using tar xzvvf nsynth_subset.tar.gz
# (this is a small subset of the "nsynth" dataset: https://magenta.tensorflow.org/datasets/nsynth)

In [92]:
pip install librosa

Note: you may need to restart the kernel to use updated packages.


In [93]:
import homework2

### Install and Load Required Libraries  

In [94]:
# !pip install librosa
# !pip install torch
# !pip install glob
# !pip install numpy

In [95]:
import torch
import torch.nn as nn
import torch.nn.functional as nnF
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import librosa
import random
import glob

In [96]:
BATCH_SIZE = 16
torch.use_deterministic_algorithms(True)

In [97]:
if not len(homework2.audio_paths):
    print("You probably need to set the dataroot folder correctly")

In [98]:
# Some helper functions. These are the same as what the autograder runs.

In [99]:
# Split dataset into train / valid / test
def split_data(waveforms, labels, train_ratio=0.7, valid_ratio=0.15):
    assert(train_ratio + valid_ratio < 1)
    test_ratio = 1 - (train_ratio + valid_ratio)
    N = len(waveforms)
    Ntrain = int(N * train_ratio)
    Nvalid = int(N * valid_ratio)
    Ntest = int(N * test_ratio)
    Wtrain = waveforms[:Ntrain]
    Wvalid = waveforms[Ntrain:Ntrain + Nvalid]
    Wtest = waveforms[Ntrain + Nvalid:]
    ytrain = labels[:Ntrain]
    yvalid = labels[Ntrain:Ntrain + Nvalid]
    ytest = labels[Ntrain + Nvalid:]
    return Wtrain,Wvalid,Wtest,ytrain,yvalid,ytest

In [100]:
def process_data(W, feature_function):
    return [feature_function(path) for path in W]

In [101]:
class InstrumentDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        features = self.features[idx]
        label = self.labels[idx]

        return features, torch.tensor(label, dtype=torch.long)

In [102]:
class Loaders():
    def __init__(self, waveforms, labels, feature_function, seed = 0):
        torch.manual_seed(seed)
        random.seed(seed)
        self.Wtrain, self.Wvalid, self.Wtest, self.ytrain, self.yvalid, self.ytest = split_data(waveforms, labels)
        
        self.Xtrain = process_data(self.Wtrain, feature_function)
        self.Xvalid = process_data(self.Wvalid, feature_function)
        self.Xtest = process_data(self.Wtest, feature_function)
        
        self.dataTrain = InstrumentDataset(self.Xtrain, self.ytrain)
        self.dataValid = InstrumentDataset(self.Xvalid, self.yvalid)
        self.dataTest = InstrumentDataset(self.Xtest, self.ytest)
        
        self.loaderTrain = DataLoader(self.dataTrain, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        self.loaderValid = DataLoader(self.dataValid, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        self.loaderTest = DataLoader(self.dataTest, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [103]:
class Pipeline():
    def __init__(self, module, learning_rate, seed = 0):
        # These two lines will (mostly) make things deterministic.
        # You're welcome to modify them to try to get a better solution.
        torch.manual_seed(seed)
        random.seed(seed)

        self.device = torch.device("cpu") # Can change this if you have a GPU, but the autograder will use CPU
        self.criterion = nn.CrossEntropyLoss()
        
        self.model = module.to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)

    def evaluate(self, loader, which = "valid"):
        self.model.eval()

        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)

                outputs = self.model(inputs)
                #loss = criterion(outputs, labels) # validation loss

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        acc = correct / total
        
        return acc
    
    def train(self, loaders,
          num_epochs=1, # Train for a single epoch by default
          model_path=None): # (Optionally) provide a path to save the best model
        val_acc = 0
        best_val_acc = 0
        for epoch in range(num_epochs):
            self.model.train()
            
            losses = []

            for inputs, labels in loaders.loaderTrain:
                inputs, labels = inputs.to(self.device), labels.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()
                losses.append(float(loss))
            
            self.model.eval()
            val_acc = self.evaluate(loaders.loaderValid)
            print("Epoch " + str(epoch) + ", loss = " + str(sum(losses)/len(losses)) +\
                  ", validation accuracy = " + str(val_acc))

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                if (model_path):
                    torch.save(self.model.state_dict(), model_path)
        print("Final validation accuracy = " + str(val_acc) + ", best = " + str(best_val_acc))
        return val_acc, best_val_acc

    def load(self, path):
        self.model.load_state_dict(torch.load(path, weights_only=True))

In [104]:
# The function below is the basis of how the autograder tests your code. Try to understand this one.

In [105]:
def test(waveforms, labels, feature_func, classifier, learning_rate, path):
    print("Extracting features...")
    test_loaders = Loaders(waveforms, labels, feature_func)
    test_pipeline = Pipeline(classifier, learning_rate)
    
    # Note: the autograder will not run this line: it will just load your saved model (next line)
    acc, best_acc = test_pipeline.train(test_loaders, 10, path)
    
    test_pipeline.load(path)
    test_acc = test_pipeline.evaluate(test_loaders.loaderTest)
    print("Test accuracy = " + str(test_acc))

In [106]:
# 1. Paths, labels, waveforms

In [107]:
# Once you've written the corresponding code in homework2.py, print these out or visualize them if you want
homework2.waveforms
homework2.labels

[0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,


In [108]:
# 2. MFCC

In [109]:
test(homework2.waveforms,
     homework2.labels,
     homework2.extract_mfcc,
     homework2.MLPClassifier(),
     0.0001,
     "best_mlp_model.weights")

Extracting features...
Epoch 0, loss = 2.0429719297422304, validation accuracy = 0.8292682926829268
Epoch 1, loss = 0.3759632180962298, validation accuracy = 0.9349593495934959
Epoch 2, loss = 0.2265150151732895, validation accuracy = 0.9512195121951219
Epoch 3, loss = 0.16812937189307478, validation accuracy = 0.975609756097561
Epoch 4, loss = 0.13555278453148073, validation accuracy = 0.975609756097561
Epoch 5, loss = 0.11442641272313064, validation accuracy = 0.975609756097561
Epoch 6, loss = 0.09867475926876068, validation accuracy = 0.983739837398374
Epoch 7, loss = 0.08646305158941282, validation accuracy = 0.983739837398374
Epoch 8, loss = 0.07663286611851719, validation accuracy = 0.983739837398374
Epoch 9, loss = 0.06844482819239299, validation accuracy = 0.983739837398374
Final validation accuracy = 0.983739837398374, best = 0.983739837398374
Test accuracy = 0.9596774193548387


In [110]:
# 3. Spectrogram

In [111]:
test(homework2.waveforms,
     homework2.labels,
     homework2.extract_spec,
     homework2.SimpleCNN(),
     0.0003,
     "best_spec_model.weights")

Extracting features...
Epoch 0, loss = 0.5862016744083829, validation accuracy = 0.9186991869918699
Epoch 1, loss = 0.5069911960098479, validation accuracy = 0.8861788617886179
Epoch 2, loss = 0.4352658796641562, validation accuracy = 0.8211382113821138
Epoch 3, loss = 0.36558789097600514, validation accuracy = 0.8943089430894309
Epoch 4, loss = 0.3084309059712622, validation accuracy = 0.9512195121951219
Epoch 5, loss = 0.26797447519169915, validation accuracy = 0.983739837398374
Epoch 6, loss = 0.23939983422557512, validation accuracy = 0.975609756097561
Epoch 7, loss = 0.21857326891687182, validation accuracy = 0.975609756097561
Epoch 8, loss = 0.20214482645193735, validation accuracy = 0.991869918699187
Epoch 9, loss = 0.18235515616834164, validation accuracy = 0.975609756097561
Final validation accuracy = 0.975609756097561, best = 0.991869918699187
Test accuracy = 0.9758064516129032


In [112]:
# 4. Mel-spectrogram

In [113]:
test(homework2.waveforms,
     homework2.labels,
     homework2.extract_mel,
     homework2.SimpleCNN(),
     0.0001,
     "best_mel_model.weights")

Extracting features...
Epoch 0, loss = 0.4973374928037326, validation accuracy = 0.8211382113821138
Epoch 1, loss = 0.3461119296650092, validation accuracy = 0.8861788617886179
Epoch 2, loss = 0.28981903443733853, validation accuracy = 0.926829268292683
Epoch 3, loss = 0.2535470492309994, validation accuracy = 0.959349593495935
Epoch 4, loss = 0.2260938651031918, validation accuracy = 0.975609756097561
Epoch 5, loss = 0.20433656519485843, validation accuracy = 0.991869918699187
Epoch 6, loss = 0.1860049106180668, validation accuracy = 1.0
Epoch 7, loss = 0.16972179379728106, validation accuracy = 1.0
Epoch 8, loss = 0.15508713697393736, validation accuracy = 1.0
Epoch 9, loss = 0.14196828193962574, validation accuracy = 1.0
Final validation accuracy = 1.0, best = 1.0
Test accuracy = 0.9596774193548387


In [114]:
# 5. Constant-Q

In [115]:
test(homework2.waveforms,
     homework2.labels,
     homework2.extract_q,
     homework2.SimpleCNN(),
     0.0001,
     "best_q_model.weights")

Extracting features...
Epoch 0, loss = 0.5682523333364062, validation accuracy = 0.7479674796747967
Epoch 1, loss = 0.4652924786011378, validation accuracy = 0.8373983739837398
Epoch 2, loss = 0.416784589489301, validation accuracy = 0.8617886178861789
Epoch 3, loss = 0.3838810014228026, validation accuracy = 0.8780487804878049
Epoch 4, loss = 0.3559975590970781, validation accuracy = 0.8780487804878049
Epoch 5, loss = 0.3288803800112671, validation accuracy = 0.9105691056910569
Epoch 6, loss = 0.3023049847947227, validation accuracy = 0.926829268292683
Epoch 7, loss = 0.27593711722228265, validation accuracy = 0.9349593495934959
Epoch 8, loss = 0.25031379693084294, validation accuracy = 0.943089430894309
Epoch 9, loss = 0.22716597384876674, validation accuracy = 0.959349593495935
Final validation accuracy = 0.959349593495935, best = 0.959349593495935
Test accuracy = 0.9838709677419355


In [116]:
# 6. Pitch shift

In [117]:
test(homework2.augmented_waveforms,
     homework2.augmented_labels,
     homework2.extract_q,
     homework2.SimpleCNN(),
     0.0001,
     "best_augmented_model.weights")

Extracting features...
Epoch 0, loss = 0.5093388832001774, validation accuracy = 0.9159891598915989
Epoch 1, loss = 0.38401881605386734, validation accuracy = 0.9376693766937669
Epoch 2, loss = 0.3338862084266212, validation accuracy = 0.940379403794038
Epoch 3, loss = 0.29694910457840673, validation accuracy = 0.959349593495935
Epoch 4, loss = 0.2679913366834323, validation accuracy = 0.9728997289972899
Epoch 5, loss = 0.24225412854165942, validation accuracy = 0.981029810298103
Epoch 6, loss = 0.21930537393523586, validation accuracy = 0.991869918699187
Epoch 7, loss = 0.19911302626132965, validation accuracy = 0.997289972899729
Epoch 8, loss = 0.18067125689790206, validation accuracy = 0.997289972899729
Epoch 9, loss = 0.16458333314706883, validation accuracy = 0.997289972899729
Final validation accuracy = 0.997289972899729, best = 0.997289972899729
Test accuracy = 0.9972972972972973


In [118]:
# 7. Extend your model to handle four classes and creatively improve its performance

In [119]:
test(homework2.waveforms,
     homework2.labels_7,
     homework2.feature_func_7,
     homework2.model_7,
     0.0003,
     "best_model_7.weights")

Extracting features...
Epoch 0, loss = 0.22950744608210194, validation accuracy = 0.9024390243902439
Epoch 1, loss = 0.20054592047300604, validation accuracy = 0.926829268292683
Epoch 2, loss = 0.1778936235027181, validation accuracy = 0.943089430894309
Epoch 3, loss = 0.15089949065198502, validation accuracy = 0.943089430894309
Epoch 4, loss = 0.15400199436893067, validation accuracy = 0.959349593495935
Epoch 5, loss = 0.1377881223646303, validation accuracy = 0.943089430894309
Epoch 6, loss = 0.14174031217892966, validation accuracy = 0.943089430894309
Epoch 7, loss = 0.11193659829182757, validation accuracy = 0.8780487804878049
Epoch 8, loss = 0.12074764225528473, validation accuracy = 0.9512195121951219
Epoch 9, loss = 0.11643354116111165, validation accuracy = 0.943089430894309
Final validation accuracy = 0.943089430894309, best = 0.959349593495935
Test accuracy = 0.9516129032258065
